# 📂 데이터셋 계층 분할 (Train / Val / Test) 및 시각화 검증 도구

이 노트북은 `data/data/dataset` 폴더 내부의 모든 표정 이미지 데이터를 머신러닝/딥러닝 학습에 적합한 표준 규격인 **Train(학습용, 80%), Val(검증용, 10%), Test(평가용, 10%)** 구조로 안전하게 분할하기 위해 개발되었습니다.

### 💡 핵심 기술 특징
1. **물리적 ImageFolder 구조화**: `data/train/감정라벨/`과 같이 폴더 구조로 분할하여 추후 프레임워크 학습 코드가 데이터를 완벽하게 구분하도록 만듭니다.
2. **계층적 분할 (Stratified Split)**: 특정 감정이 특정 세트에 몰리지 않고, 모든 세트가 균등하게 8:1:1 감정 비율을 유지하도록 합니다.
3. **안전성 (복사 방식)**: 원본 이미지를 잘라내지 않고, **복사(Copy)** 방식으로 진행하여 언제든지 원본을 안전하게 유지합니다.

## 🛠️ 1단계: 필수 라이브러리 불러오기

폴더를 만들고 파일을 복사하며, 분할 완료된 데이터 상태를 시각화하기 위한 도구들을 로드합니다.

In [ ]:
import os
import shutil
import random
from pathlib import Path
from collections import Counter
import pandas as pd
import matplotlib.pyplot as plt

# 한글 깨짐 방지 설정
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

print("✨ 분할에 필요한 모든 도구들이 준비되었습니다.")

## 🗂️ 2단계: 데이터 경로 및 감정별 파일 수집

원본 폴더에서 이미지 파일들을 파싱하여, 감정 라벨별로 파일 리스트를 안전하게 수집합니다.

In [ ]:
# 경로 설정
src_dir = Path('data/data/dataset')
base_dest_dir = Path('data')

# 지원 이미지 확장자
valid_extensions = ['.jpg', '.jpeg', '.png', '.bmp']

if not src_dir.exists():
    print(f"❌ 오류: '{src_dir}' 경로를 찾을 수 없습니다. 폴더 위치를 확인해 주세요.")
else:
    # 전체 파일 목록 가져오기
    all_files = [f for f in src_dir.iterdir() if f.is_file() and f.suffix.lower() in valid_extensions]
    
    # 감정 라벨별로 파일들을 사전(Dict) 형태로 묶기
    label_to_files = {}
    for file_path in all_files:
        label = file_path.name.split('_')[0] # 파일명에서 첫 번째 글자(감정 번호) 추출
        if label not in label_to_files:
            label_to_files[label] = []
        label_to_files[label].append(file_path)
        
    print(f"✅ 수집 완료! 감정 라벨 목록: {sorted(list(label_to_files.keys()))}")
    print(f"📈 총 이미지 개수: {len(all_files):,}장")

## 🎛️ 3단계: 대상 폴더 생성 및 고속 계층 분할 실행

설정된 8:1:1 비율에 따라 감정별 리스트를 무작위 셔플한 뒤, `train`, `val`, `test` 디렉토리에 각각 이쁘게 고속으로 복사해 배분합니다.

> **주의**: 이미지 개수가 많아 복사 연산에 약 10~30초 정도 소요될 수 있습니다. 실행하는 동안 차분히 기다려 주세요!

In [ ]:
# 1. 대상 디렉토리 구조 자동 생성
splits = ['train', 'val', 'test']
for split in splits:
    for label in label_to_files.keys():
        dest_path = base_dest_dir / split / label
        os.makedirs(dest_path, exist_ok=True)

print("📁 train/val/test 및 하위 감정 라벨 폴더 구조 생성 완료!")

# 2. 재현성을 위해 시드값 고정 후 셔플 분할 복사 진행
random.seed(42)
copy_counts = {split: Counter() for split in splits}
total_copied = 0

print("🚀 계층적 데이터 복사 연산을 시작합니다...")

for label, files in label_to_files.items():
    # 감정별 이미지 리스트 셔플
    shuffled_files = list(files)
    random.shuffle(shuffled_files)
    
    # 분할 경계 인덱스 계산 (8:1:1)
    n = len(shuffled_files)
    train_end = int(n * 0.8)
    val_end = int(n * 0.9)
    
    # 데이터 조각내기
    train_set = shuffled_files[:train_end]
    val_set = shuffled_files[train_end:val_end]
    test_set = shuffled_files[val_end:]
    
    # 고속 복사 연산 실행 (Train 세트 복사)
    for f in train_set:
        shutil.copy2(f, base_dest_dir / 'train' / label / f.name)
        copy_counts['train'][label] += 1
        total_copied += 1
        
    # Val 세트 복사
    for f in val_set:
        shutil.copy2(f, base_dest_dir / 'val' / label / f.name)
        copy_counts['val'][label] += 1
        total_copied += 1
        
    # Test 세트 복사
    for f in test_set:
        shutil.copy2(f, base_dest_dir / 'test' / label / f.name)
        copy_counts['test'][label] += 1
        total_copied += 1
        
    print(f"- [감정 라벨 {label}] 분할 복사 완료 (Train: {len(train_set)}장, Val: {len(val_set)}장, Test: {len(test_set)}장)")

print(f"\n🎉 완벽합니다! 총 {total_copied:,}장의 이미지가 train/val/test 폴더로 안전하게 복사 배분되었습니다.")

## 📊 4단계: 분할 결과 통계 데이터프레임 확인

각 세트에 감정 라벨별로 몇 장씩 나뉘었는지 이쁜 데이터 표(DataFrame)로 변환해 직관적으로 확인합니다.

In [ ]:
df_list = []
for split in splits:
    for label in sorted(label_to_files.keys()):
        count = copy_counts[split][label]
        df_list.append({'데이터셋 세트': split, '감정 라벨': label, '이미지 개수(장)': count})

df_summary = pd.DataFrame(df_list)
# 가독성을 위해 피벗 테이블 구조로 재배열
df_pivot = df_summary.pivot(index='감정 라벨', columns='데이터셋 세트', values='이미지 개수(장)')[['train', 'val', 'test']]

# 행 합계 및 열 합계 추가
df_pivot['전체 합계'] = df_pivot.sum(axis=1)
df_pivot.loc['전체 총합'] = df_pivot.sum(axis=0)

print("=== 📈 Train / Val / Test 분할 결과 집계표 ===")
df_pivot

## 📈 5단계: 누적 막대그래프(Stacked Bar Chart)를 통한 시각 검증

모든 감정 라벨들이 정확하게 8:1:1 비중을 이루고 있는지 화려한 그래프를 그려 최종 확인합니다.

In [ ]:
# 검증 그래프 작성용 피벗 테이블 재생성 (합계 행/열 제외)
df_plot = df_summary.pivot(index='감정 라벨', columns='데이터셋 세트', values='이미지 개수(장)')[['train', 'val', 'test']]

# 세련된 컬러 맵을 사용한 누적 막대그래프 시각화
ax = df_plot.plot(kind='bar', stacked=True, figsize=(11, 6), colormap='viridis', edgecolor='white', width=0.6)

plt.title('📊 데이터셋 세트별 감정 라벨 분할 및 균형 검증 차트', fontsize=16, pad=18, weight='bold')
plt.xlabel('감정 라벨 (Label)', fontsize=12, labelpad=10)
plt.ylabel('이미지 개수 (장)', fontsize=12, labelpad=10)
plt.xticks(rotation=0)
plt.legend(title='데이터셋 세트 (Set)', frameon=True, facecolor='white', edgecolor='lightgray')
plt.grid(axis='y', linestyle='--', alpha=0.5)

# 막대 위에 백분율 표시를 더해 신뢰성 및 가시성 극대화
for p in ax.patches:
    width, height = p.get_width(), p.get_height()
    x, y = p.get_xy()
    if height > 0:
        # 비율 계산하여 텍스트 얹기 (중앙 정렬)
        ax.annotate(f'{height:,}장',
                    (x + width/2, y + height/2),
                    ha='center', va='center',
                    fontsize=9, color='white', weight='bold')

plt.tight_layout()
plt.show()